In [0]:
%pip install gradio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ==============================
# Test_src.ipynb - Free Edition / UC Compatible (Safe Version)
# ==============================

# 1️⃣ Install required packages (first-time execution)
%pip install xgboost pandas numpy scikit-learn matplotlib seaborn gradio

# 2️⃣ Set project root path so Python can find src modules
import sys
project_root = '/Workspace/Users/kimjylin@gmail.com/USA-Real-Estate-Analysis'
if project_root not in sys.path:
    sys.path.append(project_root)

# 3️⃣ Import src modules

import importlib

from src import data_cleaning_pipeline
importlib.reload(data_cleaning_pipeline)

from src.data_cleaning_pipeline import clean_data
from src.features import create_features


import src.train
import src.predict

importlib.reload(src.train)
importlib.reload(src.predict)

from src.train import train_model
from src.predict import predict

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

# 4️⃣ Load Databricks tables
df_realtor = spark.table("workspace.default.realtor_data_zip")
df_cbsa = spark.table("workspace.default.OMB_cbsa_2015")

# 5️⃣ Convert Spark DataFrame to pandas
df_realtor_pd = df_realtor.toPandas()
df_cbsa_pd = df_cbsa.toPandas()

# 6️⃣ Auto-detect and rename CBSA columns
cbsa_code_col = [c for c in df_cbsa_pd.columns if 'CBSA' in c]
metromicro_col = [c for c in df_cbsa_pd.columns if 'Metropolitan' in c or 'Micropolitan' in c]

if cbsa_code_col and metromicro_col:
    df_cbsa_pd.rename(columns={
        cbsa_code_col[0]: 'cbsa_code',
        metromicro_col[0]: 'metromicro'
    }, inplace=True)
else:
    print("⚠️ Warning: CBSA columns not detected. 'metromicro' will be set to 'Unknown'.")

# 7️⃣ Build derived columns before clean_data()
# prev_sold_date to datetime
df_realtor_pd['prev_sold_date'] = pd.to_datetime(df_realtor_pd['prev_sold_date'], errors='coerce')

# season
df_realtor_pd['season'] = df_realtor_pd['prev_sold_date'].dt.month % 12 // 3 + 1
season_map = {1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Fall'}
df_realtor_pd['season'] = df_realtor_pd['season'].map(season_map)
df_realtor_pd['season'].fillna('Unknown', inplace=True)

# metromicro
if 'cbsa_code' in df_realtor_pd.columns and 'cbsa_code' in df_cbsa_pd.columns:
    df_realtor_pd = df_realtor_pd.merge(
        df_cbsa_pd[['cbsa_code', 'metromicro']],
        on='cbsa_code',
        how='left'
    )
else:
    df_realtor_pd['metromicro'] = 'Unknown'

df_realtor_pd['metromicro'].fillna('Unknown', inplace=True)

# 8️⃣ Fill numeric missing values with reasonable defaults
numeric_cols = ['bed', 'bath', 'acre_lot', 'house_size', 'zip_code', 'price']
for col in numeric_cols:
    if col in df_realtor_pd.columns:
        df_realtor_pd[col].fillna(df_realtor_pd[col].median(), inplace=True)

# 9️⃣ Data cleaning
df_clean = clean_data(df_realtor_pd)
print("✅ Data cleaning done. Sample:")
print(df_clean.head())

# 10️⃣ Feature creation
X, y = create_features(df_clean)

# Encode categorical 'Unknown' as 0
for col in ['season', 'metromicro']:
    if col in X.columns:
        X[col] = X[col].replace({'Unknown': 0}).astype(int)

print("✅ Feature creation done. Sample features:")
print(X.head())

# 11️⃣ Train the model
model = train_model(df_clean)
print("✅ Model trained successfully.")

# 12️⃣ Make predictions safely (avoid overflow)
X_eval, y_true_log = create_features(df_clean)
for col in ['season', 'metromicro']:
    if col in X_eval.columns:
        X_eval[col] = X_eval[col].replace({'Unknown': 0}).astype(int)

y_pred_log = model.predict(X_eval)
y_pred_log = np.clip(y_pred_log, a_min=None, a_max=20)  # 避免 overflow
y_pred = np.expm1(y_pred_log)

print("✅ Prediction done. Sample predictions:")
print(y_pred[:5])

# 13️⃣ Evaluate model performance
# Clip predictions
y_pred_log = model.predict(X_eval)
y_pred_log = np.clip(y_pred_log, a_min=None, a_max=20)  # 避免 overflow

# Clip true values (log1p)
y_true_log = np.log1p(df_clean['price'].values)
y_true_log = np.clip(y_true_log, a_min=None, a_max=20)  # 同樣限制上限

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
y_pred = np.expm1(y_pred_log)  # 最後轉回 dollar scale
print(f"🌟 Test RMSE (log-scale): {rmse:.4f}")


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
⚠️ Warning: CBSA columns not detected. 'metromicro' will be set to 'Unknown'.


/home/spark-90e8d58e-baf0-4062-8e8a-9b/.ipykernel/3274/command-7494460669852802-1555504713:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_realtor_pd['season'].fillna('Unknown', inplace=True)
/home/spark-90e8d58e-baf0-4062-8e8a-9b/.ipykernel/3274/command-7494460669852802-1555504713:78: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on

✅ Data cleaning done. Sample:
   brokered_by    status     price  ...  prev_sold_date  season  metromicro
0     103378.0  for_sale  105000.0  ...             NaT       0     Unknown
1      52707.0  for_sale   80000.0  ...             NaT       0     Unknown
2     103379.0  for_sale   67000.0  ...             NaT       0     Unknown
3      31239.0  for_sale  145000.0  ...             NaT       0     Unknown
4      34632.0  for_sale   65000.0  ...             NaT       0     Unknown

[5 rows x 14 columns]
✅ Feature creation done. Sample features:
   bed  bath  acre_lot  zip_code  season  metromicro
0  3.0   2.0      0.12     17361       0           0
1  4.0   2.0      0.08     17361       0           0
2  2.0   1.0      0.15     23277       0           0
3  4.0   2.0      0.10     21810       0           0
4  6.0   2.0      0.05     20197       0           0
✅ Model trained successfully.
✅ Prediction done. Sample predictions:
[4.851652e+08 4.851652e+08 4.851652e+08 4.851652e+08 4.851652e

In [0]:
import os

os.makedirs("data", exist_ok=True)

df_clean.to_csv("data/clean_realtor.csv", index=False)
